In [0]:
%pip install german-nouns

In [0]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SILVER_TABLE = "workspace.linguacopilot.silver_events"
GOLD_TABLE = "workspace.linguacopilot.gold_flashcards"
ARTICLES = {"m": "der", "f": "die", "n": "das"}


@F.pandas_udf("article string, plural string")
def noun_info(lemmas: pd.Series) -> pd.DataFrame:
    """Returns the article and plural of German nouns, using Wiktionary data.
    For compounds missing from the dictionary, uses the last component."""
    from german_nouns.lookup import Nouns

    nouns = Nouns()

    def extract(entry):
        genus = entry.get("genus") or entry.get("genus 1")
        flexion = entry.get("flexion", {})
        plural = flexion.get("nominativ plural") or flexion.get("nominativ plural 1")
        return ARTICLES.get(genus), plural

    def lookup(lemma):
        # German nouns are capitalized, other words are skipped
        if not lemma or not lemma[0].isupper():
            return (None, None)
        try:
            entries = nouns[lemma]
            if entries:
                return extract(entries[0])
            parts = nouns.parse_compound(lemma)
            if parts and len(parts) > 1:
                head = parts[-1]
                head_entries = nouns[head]
                if head_entries:
                    article, head_plural = extract(head_entries[0])
                    prefix = lemma[: len(lemma) - len(head)]
                    plural = prefix + head_plural[0].lower() + head_plural[1:] if head_plural else None
                    return (article, plural)
        except Exception as error:
            print(f"Noun lookup failed for {lemma}: {error}")
        return (None, None)

    results = [lookup(lemma) for lemma in lemmas]
    return pd.DataFrame(results, columns=["article", "plural"])


silver = spark.table(SILVER_TABLE).withColumn("lemma_key", F.lower("lemma"))

# Statistics computed on all clicks, translated or not
stats = silver.groupBy("lemma_key").agg(
    F.count("*").alias("click_count"),
    F.min("captured_at").alias("first_seen_at"),
    F.max("captured_at").alias("last_seen_at"),
)

# Example taken from the most recent fully translated click
latest_first = Window.partitionBy("lemma_key").orderBy(F.col("captured_at").desc())
examples = (
    silver
    .filter(F.col("word_translation").isNotNull() & F.col("sentence_translation").isNotNull())
    .withColumn("rank", F.row_number().over(latest_first))
    .filter(F.col("rank") == 1)
)

# Inner join: lemmas without any translated click are excluded from gold
gold = (
    examples
    .join(stats, "lemma_key", "inner")
    .withColumn("noun", noun_info(F.col("lemma")))
    .select(
        "lemma",
        F.col("noun.article").alias("article"),
        F.col("noun.plural").alias("plural"),
        F.col("word").alias("example_word_form"),
        F.col("word_translation").alias("translation"),
        F.col("sentence").alias("example_sentence"),
        F.col("sentence_translation").alias("example_sentence_translation"),
        "click_count",
        "first_seen_at",
        "last_seen_at",
        F.concat(
            F.lit("https://www.youtube.com/watch?v="),
            "video_id",
            F.lit("&t="),
            F.floor("timestamp_sec").cast("int").cast("string"),
            F.lit("s"),
        ).alias("youtube_url"),
        "source_lang",
        "target_lang",
    )
)

gold.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(GOLD_TABLE)
display(spark.table(GOLD_TABLE).orderBy(F.col("click_count").desc()))